In [3]:
# ===== Setup =====
import numpy as np
import matplotlib.pyplot as plt
import math

import tensorflow as tf
import keras
from keras import layers
from keras.layers import Dense, Input
from keras.models import Sequential

from pathlib import Path
import random
import shutil
import zipfile


vocab_size = 20000
seq_len = 200

list_of_document_strings = [
    "The invoice is due at the end of the month.",
    "Meeting notes: discuss the quarterly budget and hiring plan.",
    "Recipe for lentil soup with carrots and cumin.",
]

vectorizer = layers.TextVectorization(
    max_tokens=vocab_size, output_sequence_length=seq_len
)
vectorizer.adapt(list_of_document_strings)

shared_embedding = layers.Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True)

query_input = layers.Input(shape=(None,), name="query")
doc_input = layers.Input(shape=(None,), name="document")

_, state_h, state_c = layers.LSTM(64, return_state=True, name="encoder")(
    shared_embedding(query_input)
)

doc_repr = layers.LSTM(64, name="decoder")(
    shared_embedding(doc_input), initial_state=[state_h, state_c]
)

score = layers.Dense(1, activation="sigmoid", name="relevance")(doc_repr)

model = keras.Model([query_input, doc_input], score)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

C:\Users\willg\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\layer.py:1062: UserWarning: Layer 'encoder' (of type LSTM) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
C:\Users\willg\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\layer.py:1062: UserWarning: Layer 'decoder' (of type LSTM) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ document (InputLayer)         │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ query (InputLayer)            │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_2 (Embedding)       │ (None, None, 64)          │       1,280,000 │ query[0][0],               │
│                               │                           │                 │ document[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder (LSTM)                │ [(None, 64), (None, 64),  │          33,024 │ embedding_2[0][0]          │
│                               │ (None, 64)]               │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder (LSTM)                │ (None, 64)                │          33,024 │ embedding_2[1][0],         │
│                               │                           │                 │ encoder[0][1],             │
│                               │                           │                 │ encoder[0][2]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ relevance (Dense)             │ (None, 1)                 │              65 │ decoder[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 1,346,113 (5.14 MB)

 Trainable params: 1,346,113 (5.14 MB)

 Non-trainable params: 0 (0.00 B)